# Clearing Index Filtering

Fetch historical smoke management forecasts, merge the clearing index with your dataset, drop rows below a configurable threshold, and save the result. The workflow uses the Iowa Environmental Mesonet (IEM) AFOS archive so past dates are covered, not just the current forecast window.


**Workflow**

1. Update the configuration cell with your file paths, datetime column, target air shed, and threshold.
2. Run the helper cell to download and parse the clearing index history covering your dataset dates.
3. Merge, filter, and save the cleaned dataset.


In [ ]:
from __future__ import annotations

import re
from datetime import datetime, timedelta
from typing import Dict, Iterable, List, Optional

import pandas as pd
import requests
from zoneinfo import ZoneInfo

REQUEST_HEADERS = {"User-Agent": "Improved-Isopleths-clearing-index-notebook"}


In [ ]:
# --- User configuration ---
INPUT_DATA_PATH = "../data/utah/filtered_by_SR_data/filtered_non-max_summer"  # Path to the dataset you want to augment
DATETIME_COL = "dt"  # Column containing the local date/time information
OUTPUT_DATA_PATH = "../data/utah/filtered_by_clearing_index/SR_clearing_nonmax_summer_1000.csv"

# Air shed can be an integer 1-16 or the descriptive name from the forecast (case insensitive)
TARGET_AIRSHED = "Northern Wasatch Front"

CLEARING_INDEX_THRESHOLD = 1000  # Rows with a clearing index above this value will be dropped
LOCAL_TIMEZONE = "America/Denver"  # Time zone used by your data
REQUEST_BUFFER_DAYS = 2  # Extra days fetched on either side of the dataset window


In [ ]:
# Helper utilities for fetching and transforming clearing index history
IEM_AFOS_ENDPOINT = "https://mesonet.agron.iastate.edu/cgi-bin/afos/retrieve.py"
SMF_PIL = "SMFSLC"

AIRSHED_LOOKUP: Dict[int, str] = {
    1: "Northwest Desert",
    2: "Southwest Desert",
    3: "Virgin River",
    4: "Cache",
    5: "Northern Wasatch Front",
    6: "Wasatch Back",
    7: "Northern Slopes",
    8: "Southern Wasatch Front",
    9: "Uinta Basin",
    10: "San Pete River",
    11: "Book Cliffs",
    12: "Upper Colorado River",
    13: "Sevier River",
    14: "Lower Colorado River",
    15: "San Juan River",
    16: "Areas Above 6500 feet",
}

SAME_DAY_LABELS = {
    "TODAY",
    "TONIGHT",
    "THIS MORNING",
    "THIS AFTERNOON",
    "THIS EVENING",
    "REST OF TONIGHT",
}

NEXT_DAY_LABELS = {
    "TOMORROW",
    "TOMORROW NIGHT",
}

WEEKDAY_TO_INT = {
    "MONDAY": 0,
    "TUESDAY": 1,
    "WEDNESDAY": 2,
    "THURSDAY": 3,
    "FRIDAY": 4,
    "SATURDAY": 5,
    "SUNDAY": 6,
}

ISSUE_LINE_PATTERN = re.compile(
    r"\d{3,4}\s+[AP]M\s+[A-Za-z]{3}\s+[A-Za-z]{3}\s+[A-Za-z]{3}\s+\d{1,2}\s+\d{4}",
    re.IGNORECASE,
)
AIRSHED_BLOCK_PATTERN = re.compile(
    r"\.\.\.AIR SHED\s+(\d+)\.\.\.[\s\S]*?(?=(?:\.\.\.AIR SHED|\$\$))",
    re.MULTILINE,
)


def resolve_air_shed_id(target: int | str) -> int:
    if isinstance(target, int):
        if target in AIRSHED_LOOKUP:
            return target
        raise ValueError(f"Unknown air shed id: {target}")
    cleaned = str(target).strip().upper()
    for air_id, name in AIRSHED_LOOKUP.items():
        if name.upper() == cleaned:
            return air_id
    raise ValueError(
        f"Could not resolve air shed '{target}'. Available options: {', '.join(AIRSHED_LOOKUP.values())}"
    )


def fetch_smoke_management_text(start: datetime, end: datetime) -> str:
    params = {
        "pil": SMF_PIL,
        "fmt": "text",
        "sdate": start.strftime("%Y-%m-%dT%H:%MZ"),
        "edate": end.strftime("%Y-%m-%dT%H:%MZ"),
        "limit": 9999,
        "order": "asc",
    }
    response = requests.get(
        IEM_AFOS_ENDPOINT,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=60,
    )
    response.raise_for_status()
    text = response.text
    if "NO PRODUCTS FOUND" in text.upper():
        return ""
    return text


def split_products(raw_text: str) -> Iterable[str]:
    for chunk in raw_text.split("\x03"):
        cleaned = chunk.lstrip("\x01\n\r").strip()
        if cleaned:
            yield cleaned


def parse_issue_time(product_text: str, timezone: str) -> datetime:
    for line in product_text.splitlines():
        match = ISSUE_LINE_PATTERN.search(line)
        if match:
            parts = match.group(0).split()
            time_digits = parts[0]
            ampm = parts[1]
            date_part = " ".join(parts[3:])
            issue_naive = datetime.strptime(
                f"{date_part} {time_digits} {ampm}",
                "%a %b %d %Y %I%M %p",
            )
            return issue_naive.replace(tzinfo=ZoneInfo(timezone))
    raise ValueError("Unable to parse issue time for smoke management product.")


def parse_clearing_index_value(text_value: str) -> Optional[int]:
    cleaned = text_value.replace("+", "").strip()
    return int(cleaned) if cleaned.isdigit() else None


def normalize_period_label(label: str) -> str:
    return re.sub(r"[^A-Z ]", "", label.upper()).strip()


def resolve_valid_date(issue_dt: datetime, label: str) -> Optional[datetime.date]:
    normalized = normalize_period_label(label)
    if not normalized:
        return None
    if normalized in SAME_DAY_LABELS:
        offset = 0
    elif normalized in NEXT_DAY_LABELS:
        offset = 1
    else:
        target_weekday: Optional[int] = None
        for word in normalized.split():
            if word in WEEKDAY_TO_INT:
                target_weekday = WEEKDAY_TO_INT[word]
                break
        if target_weekday is None:
            return None
        days_ahead = (target_weekday - issue_dt.weekday()) % 7
        if days_ahead == 0:
            days_ahead = 7
        offset = days_ahead
    return (issue_dt + timedelta(days=offset)).date()


def parse_product(product_text: str, timezone: str) -> List[Dict[str, object]]:
    issue_time = parse_issue_time(product_text, timezone)
    records: List[Dict[str, object]] = []
    for match in AIRSHED_BLOCK_PATTERN.finditer(product_text):
        air_shed_id = int(match.group(1))
        block = match.group(0)
        period_label: Optional[str] = None
        for line in block.splitlines():
            stripped = line.strip()
            if not stripped:
                continue
            if stripped.endswith("..."):
                period_label = stripped.rstrip(".").strip()
                continue
            if stripped.startswith("Clearing Index") and period_label:
                ci_value = parse_clearing_index_value(stripped.split()[-1])
                if ci_value is None:
                    continue
                valid_date = resolve_valid_date(issue_time, period_label)
                if valid_date is None:
                    continue
                records.append(
                    {
                        "air_shed": air_shed_id,
                        "issue_time_local": issue_time,
                        "valid_date": valid_date,
                        "period_label": period_label,
                        "clearing_index": ci_value,
                    }
                )
    return records


def build_clearing_index_history(start: datetime, end: datetime, timezone: str) -> pd.DataFrame:
    raw_text = fetch_smoke_management_text(start, end)
    if not raw_text:
        return pd.DataFrame(
            columns=["air_shed", "issue_time_local", "valid_date", "period_label", "clearing_index"]
        )

    entries: List[Dict[str, object]] = []
    for product in split_products(raw_text):
        entries.extend(parse_product(product, timezone))

    if not entries:
        return pd.DataFrame(
            columns=["air_shed", "issue_time_local", "valid_date", "period_label", "clearing_index"]
        )

    history = pd.DataFrame(entries)
    history = history.sort_values(["air_shed", "valid_date", "issue_time_local"])
    latest = history.groupby(["air_shed", "valid_date"], as_index=False).tail(1)
    return latest.reset_index(drop=True)


In [ ]:
# Load the dataset, merge clearing index values, and filter by threshold
raw_df = pd.read_csv(INPUT_DATA_PATH)
if DATETIME_COL not in raw_df.columns:
    raise KeyError(f"Column '{DATETIME_COL}' was not found in {INPUT_DATA_PATH}.")

raw_df[DATETIME_COL] = pd.to_datetime(raw_df[DATETIME_COL], errors="coerce")
raw_df = raw_df.dropna(subset=[DATETIME_COL]).copy()

local_zone = ZoneInfo(LOCAL_TIMEZONE)
if raw_df[DATETIME_COL].dt.tz is None:
    raw_df[DATETIME_COL] = raw_df[DATETIME_COL].dt.tz_localize(
        local_zone, nonexistent="shift_forward", ambiguous="NaT"
    )
else:
    raw_df[DATETIME_COL] = raw_df[DATETIME_COL].dt.tz_convert(local_zone)
raw_df = raw_df.dropna(subset=[DATETIME_COL]).copy()

raw_df["ci_valid_date"] = raw_df[DATETIME_COL].dt.date
if raw_df["ci_valid_date"].isna().all():
    raise ValueError("No valid timestamps remain after parsing datetime information.")

target_air_shed_id = resolve_air_shed_id(TARGET_AIRSHED)

start_buffer_local = datetime.combine(raw_df["ci_valid_date"].min(), datetime.min.time()) - timedelta(days=REQUEST_BUFFER_DAYS)
end_buffer_local = datetime.combine(raw_df["ci_valid_date"].max(), datetime.max.time()) + timedelta(days=REQUEST_BUFFER_DAYS)

start_buffer_utc = start_buffer_local.replace(tzinfo=local_zone).astimezone(ZoneInfo("UTC"))
end_buffer_utc = end_buffer_local.replace(tzinfo=local_zone).astimezone(ZoneInfo("UTC"))

ci_history = build_clearing_index_history(start_buffer_utc, end_buffer_utc, LOCAL_TIMEZONE)
ci_history = ci_history[ci_history["air_shed"] == target_air_shed_id].copy()
if ci_history.empty:
    raise RuntimeError("No clearing index records were returned for the requested period and air shed.")

ci_history["ci_valid_date"] = pd.to_datetime(ci_history["valid_date"]).dt.date
ci_history = ci_history.drop_duplicates(subset=["ci_valid_date"])

merged_df = raw_df.merge(
    ci_history[["ci_valid_date", "clearing_index", "issue_time_local"]],
    on="ci_valid_date",
    how="left",
)

missing_ci = merged_df["clearing_index"].isna().sum()
if missing_ci:
    print(f"Warning: {missing_ci} rows are missing clearing index data (outside archive window or air shed mismatch).")

filtered_df = merged_df[merged_df["clearing_index"] <= CLEARING_INDEX_THRESHOLD].copy()
print(
    f"Merged rows: {len(merged_df)} | Rows at or below threshold: {len(filtered_df)} | Air shed {target_air_shed_id} ({AIRSHED_LOOKUP[target_air_shed_id]})"
)

filtered_df.head()


In [ ]:
# Persist the filtered dataset
filtered_df.to_csv(OUTPUT_DATA_PATH, index=False)
print(f"Saved {len(filtered_df)} rows to {OUTPUT_DATA_PATH}")
